# Week 6 - NLP Practical

I used the course message dataset to predict the message category from the message text.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [3]:
df = pd.read_csv("synthetic-nlp-dataset.csv")

print("Rows and columns:", df.shape)
df.head()

Rows and columns: (250, 6)


,message_id,student_key,course_week,channel,message_text,message_category
0,MSG0001,SYN023,2,course_chat,The result looks strange to me and I do not kn...,submission_deadline
1,MSG0002,SYN027,3,help_request,I have a question about the notebook; I want t...,assessment_feedback
2,MSG0003,SYN038,7,help_request,I have a question about Edge-LMS; I want to co...,submission_deadline
3,MSG0004,SYN014,6,help_request,I have a question about the project and what I...,project_group
4,MSG0005,SYN003,4,help_request,I have a question about Edge-LMS; one member c...,project_group


## Part A - Understand the data

- One row = one course message
- Text = `message_text`
- Target = `message_category`
- `message_id` and `student_key` are identifiers
- `course_week` and `channel` give extra context

In [4]:
print(df["message_category"].value_counts())

message_category
submission_deadline    50
assessment_feedback    50
project_group          50
course_content         50
technical_problem      50
Name: count, dtype: int64


In [5]:
print("Missing text:", df["message_text"].isna().sum())
print("Duplicate messages:", df["message_text"].duplicated().sum())

Missing text: 0
Duplicate messages: 40


### What I found

- The dataset has 5 categories with 50 messages in each.
- There are no missing messages.
- I found 40 duplicate messages.

## Part B - Train/test split

I used the message text as input and the message category as the target.

In [6]:
X = df["message_text"]
y = df["message_category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training:", len(X_train))
print("Testing:", len(X_test))

Training: 187
Testing: 63


## Part C - Baseline

I used DummyClassifier first as a simple baseline.

In [7]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

dummy_pred = dummy.predict(X_test)

dummy_accuracy = accuracy_score(y_test, dummy_pred)

print("Baseline accuracy:", round(dummy_accuracy, 2))

Baseline accuracy: 0.19


## Part D - TF-IDF and Logistic Regression

I used TF-IDF to change the text into numbers, then I trained Logistic Regression.

In [8]:
model = make_pipeline(
    TfidfVectorizer(),
    LogisticRegression(max_iter=1000)
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

accuracy = accuracy_score(y_test, pred)

print("Logistic Regression accuracy:", round(accuracy, 2))

Logistic Regression accuracy: 0.76


### Result

- Baseline accuracy = 19%
- Logistic Regression accuracy = 76%
- The text model performed much better than the baseline.

In [9]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, pred))

print("\nClassification Report:")
print(classification_report(y_test, pred))

Confusion Matrix:
[[11  0  2  0  0]
 [ 1  9  1  0  1]
 [ 1  2  9  0  1]
 [ 0  1  3  9  0]
 [ 1  0  1  0 10]]

Classification Report:
                     precision    recall  f1-score   support

assessment_feedback       0.79      0.85      0.81        13
     course_content       0.75      0.75      0.75        12
      project_group       0.56      0.69      0.62        13
submission_deadline       1.00      0.69      0.82        13
  technical_problem       0.83      0.83      0.83        12

           accuracy                           0.76        63
          macro avg       0.79      0.76      0.77        63
       weighted avg       0.79      0.76      0.77        63



### Evaluation

- Overall accuracy = 76%
- Most categories were predicted quite well.
- 'project_group' was the most difficult category.
- Some 'project_group' messages were confused with other categories.

## Part E - Naive Bayes

I also tested Multinomial Naive Bayes with TF-IDF.

In [10]:
nb_model = make_pipeline(
    TfidfVectorizer(),
    MultinomialNB()
)

nb_model.fit(X_train, y_train)

nb_pred = nb_model.predict(X_test)

nb_accuracy = accuracy_score(y_test, nb_pred)

print("Naive Bayes accuracy:", round(nb_accuracy, 2))

Naive Bayes accuracy: 0.76


### Result

Naive Bayes also gave 76% accuracy, so both models had the same overall accuracy.

## Part F - Count vs TF-IDF

I compared CountVectorizer and TF-IDF using the same Logistic Regression model.

In [11]:
count_model = make_pipeline(
    CountVectorizer(),
    LogisticRegression(max_iter=1000)
)

count_model.fit(X_train, y_train)
count_pred = count_model.predict(X_test)

count_accuracy = accuracy_score(y_test, count_pred)

print("Count accuracy:", round(count_accuracy, 2))
print("TF-IDF accuracy:", round(accuracy, 2))

Count accuracy: 0.84
TF-IDF accuracy: 0.76


### Result

CountVectorizer gave 84% accuracy and TF-IDF gave 76%. In this dataset, simple word counts worked better.

## Part G - Wrong predictions

I checked some messages that the model predicted incorrectly.


In [12]:
errors = pd.DataFrame({
    "Message": X_test,
    "Actual": y_test,
    "Predicted": count_pred
})

errors = errors[errors["Actual"] != errors["Predicted"]]

errors.head(5)

,Message,Actual,Predicted
20,I have a question about the notebook; I want t...,course_content,technical_problem
237,I am not sure about the upload; I am checking ...,submission_deadline,course_content
197,Could you check the project; I am unsure why w...,course_content,project_group
111,I have a question about the project and what I...,assessment_feedback,submission_deadline
159,I am stuck with this week's task and need some...,submission_deadline,project_group


### What I found

Some messages were difficult because they contained words related to different categories. Short or unclear messages were also easier to classify incorrectly.

## Part H - Unigram and bigram

I tested if adding two-word combinations changes the result.

In [13]:
bigram_model = make_pipeline(
    CountVectorizer(ngram_range=(1, 2)),
    LogisticRegression(max_iter=1000)
)

bigram_model.fit(X_train, y_train)
bigram_pred = bigram_model.predict(X_test)

bigram_accuracy = accuracy_score(y_test, bigram_pred)

print("Unigram accuracy:", round(count_accuracy, 2))
print("Unigram + bigram accuracy:", round(bigram_accuracy, 2))

Unigram accuracy: 0.84
Unigram + bigram accuracy: 0.86


### Result

Adding bigrams improved the accuracy from 84% to 86%. Two-word combinations helped a little in this dataset.

## Leakage and limitations

- I split the data before training the vectorizer and model.
- I found 40 duplicate messages. Similar messages could appear in both training and test data.
- Some messages can fit more than one category, which makes prediction difficult.

## Conclusion

- Baseline accuracy: 19%
- TF-IDF + Logistic Regression: 76%
- TF-IDF + Naive Bayes: 76%
- Count + Logistic Regression: 84%
- Count with unigrams + bigrams: 86%

The best result was 86% with CountVectorizer using unigrams and bigrams.